# Reproduce DimeNet++ Results
This notebook loads the best model checkpoint and its configuration to regenerate metrics on the QM9 test set.

In [ ]:
import os
import json
import torch
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import DimeNetPlusPlus
from tqdm.auto import tqdm

# Set device to CPU
device = torch.device('cpu')
print(f"Using device: {device}")

## 1. Load Configuration and Initialize Model

In [ ]:
config_path = 'dimenet_config.json'
model_path = 'dimenet_best_model.pt'

if not os.path.exists(config_path) or not os.path.exists(model_path):
    raise FileNotFoundError("Required model or config file missing. Please run training first.")

with open(config_path, 'r') as f:
    config = json.load(f)

model = DimeNetPlusPlus(
    hidden_channels=config["hidden_channels"],
    out_channels=config["out_channels"],
    num_blocks=config["num_blocks"],
    num_spherical=config["num_spherical"],
    num_radial=config["num_radial"],
    cutoff=config["cutoff"],
    int_emb_size=config["int_emb_size"],
    out_emb_channels=config["out_emb_channels"],
    basis_emb_size=config["basis_emb_size"]
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model and configuration loaded successfully.")

## 2. Load Dataset

In [ ]:
dataset = QM9(root='./data/QM9')
test_dataset = dataset[120000:]
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

target_mean = config['target_mean']
target_std = config['target_std']
TARGET_IDX = config['target_idx']

def denormalize(y_norm):
    return y_norm * target_std + target_mean

## 3. Evaluate and Report

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    total_mae = 0
    for batch in tqdm(loader, desc="Evaluating"):
        batch = batch.to(device)
        out = model(batch.z, batch.pos, batch.batch)
        
        preds = denormalize(out.squeeze())
        labels = batch.y[:, TARGET_IDX]
        total_mae += (preds - labels).abs().sum().item()
    return total_mae / len(loader.dataset)

test_mae = evaluate(model, test_loader)
print(f"Reproduced Test MAE: {test_mae * 1000:.2f} meV")